# Project K, tracing round 1: Qwen2.5-Omni-3B

**Why this notebook exists.** Every Project K test so far
read the model's answers from outside. The answers say what
the model reported; they cannot say where, inside the model,
a sound stopped being carried. This notebook opens the model
and walks the sound through it. Qwen2.5-Omni-3B goes first
because its insides are standard: every layer's numbers (its
hidden states) can be saved while it listens.

**The goal.** Find the layer where the loud tone fades
inside the model, and test whether the speech is ever lost
at all. Round 1 carries two instruments: the helpers
(per-layer probes) and the attention watch. The third
instrument from the plan, surgery (activation patching), is
NOT in this notebook; it is the follow-up build, and it is
the causal check on whatever this round finds.

**The helpers, in plain words.** A helper is a tiny
classifier we train ourselves, on one layer's numbers only.
We play clips we chose, so we always know which clips carry
the sound and which do not. For each layer, the helper
learns the shape that layer's numbers take when the sound
is in them. Then it walks the layers on clips it never saw
in training. The layer where the helper stops finding the
sound is the neighborhood where the sound faded. The helper
only reads; it never changes the model, and the model is
never taught anything.

**What runs.** 144 short clips, generated inside the runner
with a fixed seed, each 4.0 seconds: 48 tone clips (a short
1000 Hz burst, pitch varied across 800, 1000, and 1200 Hz,
placed at different times and levels), 48 speech clips (the
one probe sentence, "The blue kettle boils at seven forty
two.", shifted in time and gain, the documented
augmentation for having a single sentence), and 48 negative
clips shared by both tasks (12 digital silence, 18 white
noise bursts, 18 low 120 Hz rumbles). Sound alone: no
picture rides along in this round, because the Series Two
2x2 showed the still picture is itself an interference
source for this model.

**The sanity gate comes first.** Before any clip set is
built, one tone clip goes through the model with its hidden
states saved. The runner finds the audio-token positions in
the model's input from the model's own tokenizer and config
(never a guessed number), prints the layer count, the
hidden size, the sequence length, and the audio-token
count, and stops dead if the audio-token count is zero.
Every later number depends on those positions being right,
so the gate is loud and early by design.

**Readout rules, fixed in advance** (from the tracing
plan):

- A helper failing at a layer means "not readable by this
  helper," which can also mean the information changed
  shape. No stage is declared lost on the helpers alone;
  the surgery build is the causal check.
- The shuffled-label control must sit at chance (about
  0.50). A control that beats chance means the helper is
  cheating on some side pattern, and the run is void until
  fixed.
- Predictions, registered before the run: the tone stays
  readable into the middle layers, then fades; the
  attention the answer pays to the tone's tokens is near
  zero; the speech stays readable at every layer, because
  its transcription is perfect, so its denial lives in how
  the answer is formed, past the information.

**Where it runs.** On the free Colab account (kadjei1),
under the routing rule: the 3B is a small model. **How to
run it.** Run the cells from the top, in order. Step 1 sets
up and proves the stack. Step 2 gets the three master
sounds. Step 3 writes the runner and runs its check mode
(proof, loading candidates, the sanity gate). Step 4 runs
battery mode: clip generation, extraction, the helpers, the
attention readout, and a plain-language tail.

**The one rule.** We claim only what we ran: these sounds,
this model, these layers, this run.

## Step 1 · Setup

The next cell installs the parts the model needs, with fixed version numbers
so the run is the same every time. It then prints the graphics chip it found.

**Before you run it:** the chip must be a T4. In the Colab menu, open
Runtime, then "Change runtime type", and pick the T4 GPU. On the free plan
this is the normal choice. If the cell prints "no GPU found", stop and change
the runtime type first. The model will not fit in plain memory.

The install takes a few minutes. That is normal.

One more setup step, new after run 2: the first code cell sets a memory
option (expandable segments) before torch loads. It lets the chip's memory
manager grow in pieces instead of reserving one big block, which helps when
the model already takes most of the chip. If you rerun cells in an old
session, restart the runtime once first, so the option takes effect from a
clean start.
**Tracing addition.** At the end, the cell proves the
stack it just installed: it prints the imported
transformers version and the file it loaded from, and
stops the run if the version is not the pinned 4.52.4.
It also proves scikit-learn, the library the helper
classifiers are built with: if it is missing, the cell
installs it, and either way it prints the version it
found. This notebook's model runs under the session's
own Python, so the versions printed here are the run's
versions.


In [ ]:
# --- Setup -------------------------------------------------------------------
# First thing in this cell, before torch is loaded: ask the memory manager
# to use growing segments. This must be set BEFORE torch loads, which is
# why it sits above the torch import below. In a fresh runtime it takes
# effect. If you rerun this cell in a session where torch was already
# loaded, restart the runtime once and run from the top.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Fixed versions, so today's run matches tomorrow's run.
# soundfile and librosa are how the model's tools read the sound track
# out of each video file. Without them, the video would arrive silent,
# and this whole test would prove nothing.
import sys, subprocess

PACKAGES = [
    "transformers==4.52.4",
    "accelerate==1.8.1",
    "qwen-omni-utils==0.0.8",
    "soundfile==0.13.1",
    "librosa==0.11.0",
    "av",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PACKAGES,
               check=False)
print("install step done")

import torch
if torch.cuda.is_available():
    print("GPU found:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("no GPU found. Stop here. Set the runtime type to the T4 GPU,")
    print("then run this cell again.")
# --- Tracing addition: prove the installed stack --------------------------
# The model and the probes run under this same session
# Python, so the versions installed above are the versions
# the run uses. Prove them here, and stop if the
# transformers pin is not met.
import transformers
print("transformers version:", transformers.__version__)
print("transformers file:", transformers.__file__)
if transformers.__version__ != "4.52.4":
    raise SystemExit("STOP: transformers is "
                     + transformers.__version__
                     + ", not the pinned 4.52.4. Do not go on.")
print("stack proof passed: transformers 4.52.4, the Series One pin.")

# The helper classifiers are scikit-learn logistic
# regressions. Colab sessions normally ship scikit-learn;
# install it only if it is missing, and print the version
# either way so the run record names what trained the
# helpers.
try:
    import sklearn
    print("scikit-learn version:", sklearn.__version__)
except ImportError:
    print("scikit-learn missing; installing scikit-learn==1.6.1")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "scikit-learn==1.6.1"], check=False)
    import sklearn
    print("scikit-learn version:", sklearn.__version__)


## Step 2 · Get the three master sounds

Everything this run builds its clips from lives in your
Google Drive, in the folder named `codec-control`, the
codec control's folder. The next cell links your Drive,
finds that folder, and copies three files to a plain work
folder here:

- `codec_speech_raw.wav`: the probe sentence master, the
  swap study's own speech track, uncompressed.
- `codec_tone_ref_raw.wav`: the loud 1000 Hz tone master,
  the burst at the reference level, uncompressed.
- `codec_silence_raw.wav`: the silence master, true digital
  zeros, uncompressed.

The speech master is the source of all 48 speech clips; the
tone master is the sanity gate's clip; the silence master
is the record's proof of what the negative pool's silence
is. The runner re-measures everything from inside the
session before it believes any of it.

Do not go on until all three files say "ready". If any file
is missing, the cell stops and names it.

In [ ]:
# --- Get the three master sounds ---------------------------------------------------
import os, shutil, subprocess

FILES = ["codec_speech_raw.wav", "codec_tone_ref_raw.wav",
         "codec_silence_raw.wav"]
WORK = "/content/clips"
DRIVE_CLIPS = "/content/drive/MyDrive/codec-control"
os.makedirs(WORK, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped:", e)

SOURCES = [DRIVE_CLIPS, "/content/codec-control"]

def bring(name):
    dest = os.path.join(WORK, name)
    if os.path.exists(dest):
        return True
    for folder in SOURCES:
        src = os.path.join(folder, name)
        if os.path.exists(src):
            shutil.copy(src, dest)
            print("copied", name, "from", folder)
            return True
    return False

def duration_s(path):
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "csv=p=0", path], capture_output=True, text=True)
    try:
        return float(out.stdout.strip())
    except Exception:
        return None

missing = []
for name in FILES:
    bring(name)
    path = os.path.join(WORK, name)
    if os.path.exists(path):
        d = duration_s(path)
        extra = f", {d:.3f} s" if d is not None else ""
        print("ready:  ", name, f"({os.path.getsize(path) / 1e6:.2f} MB{extra})")
    else:
        missing.append(name)
        print("MISSING:", name)
if missing:
    print()
    print("!" * 72)
    print("STOP. These master files are missing:")
    for name in missing:
        print("   ", name)
    print("Check the Drive folder name is exactly codec-control, or")
    print("upload the files by hand, then run this cell again.")
    print("Do not go on to Step 3.")
    print("!" * 72)
    raise SystemExit("STOP: master files missing in Step 2")
print()
print("ready: all 3 masters (speech, tone, silence)")

## Step 3 · Write the runner, then check: proof, candidates, sanity gate

The model never runs inside this notebook's cells directly.
It runs in a plain script, `/content/qwen_tracing_run.py`,
under this session's own Python, the Qwen notebooks'
pattern. The first cell below writes that script. The
second runs it in **check mode**.

Check mode does three things, in order:

1. **Proof again.** The versions and files the runner
   imported, with the transformers pin checked a second
   time: exactly 4.52.4, or the run stops. The scikit-learn
   version prints here too, and one note explains the single
   deliberate difference from the earlier runners' load
   call: the candidates load with eager attention, because
   this build must return attention weights and the default
   sdpa kernel does not.
2. **The loading candidates.** The codec runner's list: the
   3B in 16-bit form (float16) first, the 7B AWQ build
   second, only if the small one fails. The talker is turned
   off on whichever loads. A failed candidate prints its
   full traceback and appends the same words to
   `tracing_qwen3b_load_log.txt` in the tracing folder.
3. **The sanity gate.** One tone clip (the tone master from
   Step 2) goes through the model's thinker with hidden
   states saved. The runner searches the model's own
   tokenizer and config for the audio token, prints every
   candidate with its decoded form and its count inside the
   gate input, and takes its positions from the audio pad
   token that actually occurs there. It prints the layer
   count, the hidden size, the sequence length, and the
   audio-token count, and stops dead (exit code 8) if the
   count is zero or the hidden states are missing. No clips
   are generated and no extraction runs in check mode.

In [ ]:
# --- Write the tracing runner script --------------------------------------------
# The runner holds the proof, the loading candidates, the
# sanity gate, the clip generation, the extraction, the
# helpers, and the attention readout. It is written to a
# plain file here and only ever run under this session's
# own Python, the interpreter Step 1 installed into.
RUNNER_SOURCE = '\n# qwen_tracing_run.py - the tracing round 1 runner for\n# Qwen2.5-Omni-3B. Written by project_k_qwen_tracing_colab.ipynb.\n# It runs under the notebook session\'s own Python: the Qwen\n# notebooks install their stack into the session (Step 1), and\n# there is no private environment to run under. Modes: "check"\n# runs the proof, the loading candidates, and the sanity gate,\n# then stops. "battery" does all of that and then generates\n# the probe clips, extracts the per-layer features, trains the\n# helpers, reads the attention subset, and saves every output.\n#\n# Exit codes, one per failure family, all loud:\n#   3  the stack proof failed (transformers is not 4.52.4)\n#   4  both loading candidates failed\n#   7  a feed returned no audio (never padded, never scored)\n#   8  the sanity gate failed: no audio tokens found, or no\n#      hidden states came back\n#   9  probe clip generation failed its own measurements\n#   10 the extraction count does not match the manifest count\nimport os\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\nimport sys\n\nMODE = sys.argv[1] if len(sys.argv) > 1 else "check"\nprint("runner mode:", MODE)\nprint("runner python:", sys.version.split()[0], sys.executable)\n\n# --- Proof lines, printed before anything loads --------------------------------\nimport torch, transformers\nprint("torch version:", torch.__version__)\nprint("transformers version:", transformers.__version__)\nprint("transformers file:", transformers.__file__)\nimport qwen_omni_utils\nprint("qwen_omni_utils file:", qwen_omni_utils.__file__)\nif transformers.__version__ != "4.52.4":\n    print("STOP: this script is running on transformers",\n          transformers.__version__, "and needs exactly 4.52.4,")\n    print("the version the Series One Qwen notebooks pin.")\n    sys.exit(3)\nimport sklearn\nprint("scikit-learn version:", sklearn.__version__)\nprint("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()\n      else "none")\nprint("attention implementation note: the loading candidates below pass")\nprint("attn_implementation=\'eager\'. The attention readout is a required")\nprint("output of this build, and the default sdpa kernel does not return")\nprint("attention weights. The sequences here are short, so the memory")\nprint("cost is small, and the hidden-state math is the same.")\n\n# --- Homes for the outputs -------------------------------------------------------\n# The tracing outputs live in a Drive folder named\n# tracing-qwen3b, created here if it is missing, with /content\n# as the second home, the house pattern.\nDRIVE_TRACING = "/content/drive/MyDrive/tracing-qwen3b"\nLOCAL_TRACING = "/content/tracing-qwen3b"\nfor d in (DRIVE_TRACING, LOCAL_TRACING):\n    try:\n        os.makedirs(d, exist_ok=True)\n    except Exception:\n        pass\n\ndef out_paths(name):\n    return [os.path.join(DRIVE_TRACING, name),\n            os.path.join(LOCAL_TRACING, name)]\n\ndef save_both(name, write_fn):\n    saved = []\n    for p in out_paths(name):\n        try:\n            write_fn(p)\n            saved.append(p)\n            print("saved:", p)\n        except Exception as e:\n            print("could not save", p, "-", str(e)[:160])\n    return saved\n\n# --- The load log ------------------------------------------------------------------\nimport traceback, datetime\n\nLOAD_LOG_PATHS = out_paths("tracing_qwen3b_load_log.txt")\n\ndef log_load(text):\n    for p in LOAD_LOG_PATHS:\n        try:\n            with open(p, "a") as fh:\n                fh.write(text)\n        except Exception:\n            pass   # that home is not mounted; the printed copy stands\n\ndef log_paths_saved():\n    return [p for p in LOAD_LOG_PATHS if os.path.exists(p)]\n\nlog_load("=" * 70 + "\\n")\nlog_load("Qwen2.5-Omni-3B tracing round 1 load log - "\n         + datetime.datetime.now().isoformat(timespec="seconds")\n         + " - runner mode: " + MODE + "\\n")\n\n# --- Load the model: the codec runner\'s candidates ---------------------------------\n# Same candidates, same order, same talker-off call. The one\n# printed difference is attn_implementation="eager" (see the\n# proof lines above for why).\nfrom transformers import (Qwen2_5OmniForConditionalGeneration,\n                          Qwen2_5OmniProcessor)\n\ndef free_memory():\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\nmodel = None\nprocessor = None\nMODEL_NAME = None   # repo name of the model that loaded\nLOAD_MODE = None    # the fit that worked\n\nCANDIDATES = [("Qwen/Qwen2.5-Omni-3B", "fp16"),\n              ("Qwen/Qwen2.5-Omni-7B-AWQ", "AWQ weights, fp16 load")]\n\ncandidate_errors = []\nfor repo, mode in CANDIDATES:\n    print("=" * 70)\n    print(f"candidate: {repo} ({mode})")\n    try:\n        m = Qwen2_5OmniForConditionalGeneration.from_pretrained(\n            repo, torch_dtype=torch.float16, device_map="auto",\n            attn_implementation="eager")\n        m.disable_talker()  # written answers only; saves memory\n        p = Qwen2_5OmniProcessor.from_pretrained(repo)\n        model, processor = m, p\n        MODEL_NAME, LOAD_MODE = repo, mode\n        print("loaded:", repo)\n        break\n    except Exception:\n        tb = traceback.format_exc()\n        candidate_errors.append((repo, mode, tb))\n        print("this candidate failed. Full traceback:")\n        print(tb)\n        log_load("-" * 70 + "\\n")\n        log_load("candidate failed: " + repo + " (" + mode + ")\\n")\n        log_load(tb)\n        model = None\n        processor = None\n        free_memory()\n\nprint()\nif model is None:\n    print("!" * 70)\n    print("BOTH CANDIDATES FAILED. No model loaded.")\n    saved = log_paths_saved()\n    if saved:\n        print("The full traceback of every candidate is saved, word")\n        print("for word, in:")\n        for p in saved:\n            print("   ", p)\n    else:\n        print("The load log could not be written anywhere; the full")\n        print("tracebacks are in the output above, and nowhere else.")\n    print("Do not go on. Show this whole output to Muse.")\n    print("!" * 70)\n    sys.exit(4)\nprint("MODEL FOR THIS RUN:", MODEL_NAME, "in", LOAD_MODE, "form")\nlog_load("loaded: " + MODEL_NAME + " in " + LOAD_MODE + " form\\n")\nprint("environment: Colab session python (Step 1 installs),",\n      "transformers 4.52.4")\nif torch.cuda.is_available():\n    used = torch.cuda.memory_allocated() / 1e9\n    print(f"chip memory already used by the model: {used:.1f} GB")\n\nTHINKER = model.thinker\nprint("thinker:", type(THINKER).__name__)\n\n# --- The feed path -----------------------------------------------------------------\n# Sound alone, the codec control\'s Arm B form minus the\n# picture: the conversation carries an audio element naming\n# the WAV file and a text element, and no image and no video\n# element. process_mm_info decodes the audio element with the\n# same librosa call at 16 kHz mono it uses for a video\n# soundtrack. The text is the Series Two presence question\n# for the clip\'s sound, fixed per class: the tracing walks\n# the model\'s insides while it hears the clip under the same\n# question the outside tests asked. Negatives carry the tone\n# question, the primary task\'s question, and the manifest\n# records the exact prompt on every clip.\nimport json, math\nimport numpy as np\nimport librosa\nimport soundfile as sf\nfrom qwen_omni_utils import process_mm_info\n\nCLIPS = "/content/clips"\nPROMPTS = {\n    "tone": ("Is there the sound of a tone in this clip? "\n             "Answer yes or no."),\n    "speech": ("Is there the sound of speech in this clip? "\n               "Answer yes or no."),\n    "negative": ("Is there the sound of a tone in this clip? "\n                 "Answer yes or no."),\n}\n\nclass FeedError(RuntimeError):\n    pass\n\nclass GateError(RuntimeError):\n    pass\n\ndef conversation_for(path, prompt):\n    return [{\n        "role": "user",\n        "content": [\n            {"type": "audio", "audio": path},\n            {"type": "text", "text": prompt},\n        ],\n    }]\n\ndef feed_inputs(path, prompt, expect_silence=False):\n    # Run the feed path for one clip and build the model\'s\n    # inputs. Raises FeedError if the path returns no audio:\n    # a missing track stops the run, it is never padded. A\n    # track that decodes to all zeros also stops the run,\n    # unless the caller declares the clip silent by design\n    # (the generated digital-silence negatives): an all-zeros\n    # feed on a clip that should carry sound is a broken\n    # feed, not a quiet clip, and scoring it anyway is how\n    # silent failures get into the record.\n    if not os.path.exists(path):\n        raise FeedError(f"feed file missing: {path}")\n    conversation = conversation_for(path, prompt)\n    audios, images, videos = process_mm_info(\n        conversation, use_audio_in_video=False)\n    if audios is None or len(audios) == 0 or audios[0] is None:\n        raise FeedError(\n            f"the feed path returned no audio for {path}")\n    audio_np = np.asarray(audios[0], dtype=np.float32)\n    if len(audio_np) == 0:\n        raise FeedError(\n            f"decoded track is empty for {path}")\n    if not expect_silence and not np.any(audio_np != 0):\n        raise FeedError(\n            f"decoded track is all zeros for {path}, on a clip "\n            f"that should carry sound")\n    text = processor.apply_chat_template(\n        conversation, add_generation_prompt=True, tokenize=False)\n    inputs = processor(text=text, audio=audios,\n                       return_tensors="pt", padding=True)\n    inputs = inputs.to(model.device).to(model.dtype)\n    return inputs, audio_np\n\ndef thinker_forward(inputs, want_attentions=False):\n    # One frozen pass through the thinker with every layer\'s\n    # numbers saved. The model is never trained, never asked\n    # to generate; this is a reading, not a conversation.\n    with torch.no_grad():\n        out = THINKER(**inputs, output_hidden_states=True,\n                      output_attentions=want_attentions)\n    return out\n\n# --- Find the audio tokens -----------------------------------------------------------\n# The audio-token id is discovered, never guessed. Evidence,\n# all printed: the tokenizer\'s tokens whose text mentions\n# audio; the config attributes on the model config and the\n# thinker config; and the count of each candidate inside a\n# real input_ids row. The chosen id must decode to the audio\n# pad token and must actually occur in the input. Anything\n# less stops the run at the gate.\ndef find_audio_token_id(input_ids_row):\n    tok = processor.tokenizer\n    print("-" * 70)\n    print("AUDIO TOKEN SEARCH")\n    vocab = tok.get_vocab()\n    audio_named = {t: i for t, i in vocab.items() if "audio" in t}\n    for t in sorted(audio_named):\n        print(f"  tokenizer token {t!r} -> id {audio_named[t]}")\n    config_ids = {}\n    for holder_name, holder in (\n            ("model.config", model.config),\n            ("thinker_config",\n             getattr(model.config, "thinker_config", None))):\n        if holder is None:\n            continue\n        for attr in ("audio_token_id", "audio_token_index"):\n            val = getattr(holder, attr, None)\n            if val is not None:\n                config_ids[f"{holder_name}.{attr}"] = int(val)\n                print(f"  {holder_name}.{attr} = {val}")\n    candidates = {}\n    pad_id = tok.convert_tokens_to_ids("<|audio_pad|>")\n    if pad_id is not None and int(pad_id) >= 0:\n        candidates[int(pad_id)] = "<|audio_pad|> (converted by name)"\n    for t, i in audio_named.items():\n        candidates[int(i)] = f"tokenizer token {t!r}"\n    for where, val in config_ids.items():\n        candidates.setdefault(val, f"config {where}")\n    row = [int(x) for x in input_ids_row]\n    scored = []\n    for cid, why in sorted(candidates.items()):\n        count = row.count(cid)\n        decoded = tok.decode([cid])\n        print(f"  candidate id {cid} ({why}): decodes to "\n              f"{decoded!r}, occurs {count} time(s) in the "\n              f"sanity input")\n        scored.append((cid, why, decoded, count))\n    # The pad token first: it is the placeholder the\n    # processor expands into the sequence where the audio\n    # sits, so its positions ARE the audio-token positions.\n    for cid, why, decoded, count in scored:\n        if decoded == "<|audio_pad|>" and count > 0:\n            print(f"AUDIO TOKEN ID: {cid} ({why}), decodes to "\n                  f"{decoded!r}, {count} position(s) in the input")\n            return cid, count\n    for cid, why, decoded, count in scored:\n        if "audio" in decoded and count > 0:\n            print(f"AUDIO TOKEN ID: {cid} ({why}), decodes to "\n                  f"{decoded!r}, {count} position(s) in the input")\n            return cid, count\n    raise GateError(\n        "no candidate audio token occurs in the sanity input; "\n        f"candidates were {[(c, d, n) for c, _, d, n in scored]}")\n\nAUDIO_TOKEN_ID = None   # set by the sanity gate\n\ndef audio_positions(input_ids_row):\n    row = [int(x) for x in input_ids_row]\n    return [i for i, x in enumerate(row) if x == AUDIO_TOKEN_ID]\n\n# --- The sanity gate -------------------------------------------------------------------\n# One tone clip through the thinker, before anything else is\n# built. Prints the shape of what came back and stops dead\n# (exit 8) if the audio tokens are not found or the hidden\n# states are missing. Loud and early, by design: every later\n# number in this run depends on these positions being right.\ndef sanity_gate():\n    global AUDIO_TOKEN_ID\n    tone_path = os.path.join(CLIPS, "codec_tone_ref_raw.wav")\n    print("=" * 70)\n    print("SANITY GATE: one tone clip through the thinker")\n    print("gate clip:", tone_path)\n    try:\n        inputs, audio_np = feed_inputs(tone_path, PROMPTS["tone"])\n    except FeedError as e:\n        print()\n        print("!" * 70)\n        print("SANITY GATE FEED FAILURE:")\n        print(str(e))\n        print("The gate clip\'s feed returned no audio. No extraction")\n        print("has run. Show this whole output to Muse.")\n        print("!" * 70)\n        sys.exit(7)\n    print(f"gate feed: decoded waveform {len(audio_np)} samples "\n          f"at 16 kHz ({len(audio_np) / 16000:.2f} s)")\n    row = inputs.input_ids[0].tolist()\n    try:\n        AUDIO_TOKEN_ID, gate_count = find_audio_token_id(row)\n    except GateError as e:\n        print()\n        print("!" * 70)\n        print("SANITY GATE FAILED:")\n        print(str(e))\n        print("Without the audio-token positions, every later number")\n        print("in this run would be built on sand. No extraction has")\n        print("run. Show this whole output to Muse.")\n        print("!" * 70)\n        sys.exit(8)\n    out = thinker_forward(inputs)\n    hs = getattr(out, "hidden_states", None)\n    if hs is None or len(hs) == 0:\n        print()\n        print("!" * 70)\n        print("SANITY GATE FAILED: the thinker returned no hidden")\n        print("states. No extraction has run. Show this whole output")\n        print("to Muse.")\n        print("!" * 70)\n        sys.exit(8)\n    n_states = len(hs)\n    seq_len = int(hs[0].shape[1])\n    hidden = int(hs[0].shape[2])\n    print(f"hidden states returned: {n_states} tensors "\n          f"(layer 0 is the embedding output, then one per "\n          f"thinker layer)")\n    print(f"thinker layers: {n_states - 1}")\n    print(f"hidden size: {hidden}")\n    print(f"sequence length: {seq_len}")\n    print(f"audio-token count: {gate_count}")\n    print(f"audio-token positions: first "\n          f"{audio_positions(row)[:5]}, last "\n          f"{audio_positions(row)[-5:]}")\n    if gate_count == 0:\n        print()\n        print("!" * 70)\n        print("SANITY GATE FAILED: audio-token count is zero.")\n        print("!" * 70)\n        sys.exit(8)\n    print("SANITY GATE PASSED: the feed carries audio, the thinker")\n    print("returns its layers, and the audio tokens are located and")\n    print("counted. The positions every later number uses are these.")\n    del out, inputs\n    free_memory()\n    return n_states, hidden\n\nN_STATES, HIDDEN_SIZE = sanity_gate()\n\nif MODE == "check":\n    print()\n    print("check mode done: proof, candidates, and the sanity gate")\n    print("all passed. No clips were generated and no extraction ran.")\n    print("Run Step 4 for the battery.")\n    sys.exit(0)\n\n# --- Probe clip generation -------------------------------------------------------------\n# Fixed seed, so the run can be repeated exactly and the\n# split can be re-derived. 144 clips, each exactly 4.0 s at\n# 16 kHz mono (64000 samples). Tone positives vary pitch,\n# placement, and level. Speech positives are the one probe\n# sentence (the master\'s active region), shifted in time and\n# gain: the documented augmentation for having a single\n# sentence. Negatives are a shared pool: digital silence,\n# white-noise bursts, low sine rumble. Every clip is measured\n# (sample count, RMS) and printed before anything uses it,\n# and a failed measurement stops the run (exit 9).\nimport csv, io\n\nSEED = 20261103\nSR = 16000\nCLIP_SAMPLES = SR * 4\nCLIP_DIRS = ["/content/drive/MyDrive/tracing-qwen3b/tracing_clips",\n             "/content/tracing_clips"]\nfor d in CLIP_DIRS:\n    try:\n        os.makedirs(d, exist_ok=True)\n    except Exception:\n        pass\nMANIFEST_FIELDS = [\'clip_name\', \'group\', \'file\', \'freq_hz\', \'start_s\', \'burst_s\', \'level_dbfs\', \'gain_db\', \'prompt\']\n\ndef rms_db_np(x):\n    if len(x) == 0:\n        return -120.0\n    v = float(np.sqrt(np.mean(np.square(x.astype(np.float64)))))\n    return 20.0 * math.log10(v) if v > 0 else -120.0\n\ndef db_to_amp(db):\n    return float(10.0 ** (db / 20.0))\n\ndef write_clip(name, wave):\n    wave = np.asarray(wave, dtype=np.float32)\n    assert len(wave) == CLIP_SAMPLES, \\\n        f"clip {name} has {len(wave)} samples, not {CLIP_SAMPLES}"\n    saved = []\n    for d in CLIP_DIRS:\n        p = os.path.join(d, name)\n        try:\n            if not os.path.exists(p):\n                sf.write(p, wave, SR, subtype="PCM_16")\n            saved.append(p)\n        except Exception:\n            pass\n    # The extraction reads from /content first: a plain local\n    # path the feed cannot miss.\n    local = os.path.join("/content/tracing_clips", name)\n    return local\n\ndef speech_segment():\n    # The sentence inside the speech master: the region whose\n    # short-window RMS rises above -45 dBFS, with a small pad.\n    path = os.path.join(CLIPS, "codec_speech_raw.wav")\n    wave, _ = librosa.load(path, sr=SR, mono=True)\n    wave = np.asarray(wave, dtype=np.float32)\n    win = SR // 100  # 10 ms\n    n = len(wave) // win\n    rms = np.array([rms_db_np(wave[i * win:(i + 1) * win])\n                    for i in range(n)])\n    above = np.where(rms > -45.0)[0]\n    if len(above) == 0:\n        raise RuntimeError(\n            "no active region found in the speech master; the "\n            "speech positives cannot be built")\n    start = max(0, int(above[0] * win) - SR // 20)\n    end = min(len(wave), int((above[-1] + 1) * win) + SR // 20)\n    seg = wave[start:end]\n    print(f"speech segment: {len(seg) / SR:.2f} s cut from the "\n          f"master at {start / SR:.2f} s to {end / SR:.2f} s")\n    if len(seg) > CLIP_SAMPLES:\n        seg = seg[:CLIP_SAMPLES]\n        print("speech segment trimmed to the 4.0 s window")\n    return seg\n\ndef generate_clips():\n    rng = np.random.default_rng(SEED)\n    rows = []\n    # ---- tone positives (48)\n    for i in range(48):\n        freq = (800, 1000, 1200)[i % 3]\n        start_s = float(rng.uniform(0.5, 3.0))\n        level = float(rng.uniform(-26.0, -14.0))\n        burst_s = 0.5\n        wave = np.zeros(CLIP_SAMPLES, dtype=np.float32)\n        n_burst = int(burst_s * SR)\n        t = np.arange(n_burst) / SR\n        burst = (db_to_amp(level)\n                 * np.sin(2 * np.pi * freq * t)).astype(np.float32)\n        edge = SR // 200  # 5 ms raised-cosine edges, no clicks\n        ramp = 0.5 - 0.5 * np.cos(\n            np.linspace(0, np.pi, edge)).astype(np.float32)\n        burst[:edge] *= ramp\n        burst[-edge:] *= ramp[::-1]\n        s0 = int(start_s * SR)\n        wave[s0:s0 + n_burst] += burst\n        name = f"tone_{i:02d}.wav"\n        path = write_clip(name, wave)\n        rows.append({\n            "clip_name": name[:-4], "group": "tone", "file": path,\n            "freq_hz": freq, "start_s": round(start_s, 3),\n            "burst_s": burst_s, "level_dbfs": round(level, 2),\n            "gain_db": "", "prompt": PROMPTS["tone"],\n            "_wave": wave})\n    # ---- speech positives (48)\n    seg = speech_segment()\n    max_offset = max(0.0, min(1.5, 4.0 - len(seg) / SR))\n    for i in range(48):\n        offset = float(rng.uniform(0.0, max_offset)) \\\n            if max_offset > 0 else 0.0\n        gain = float(rng.uniform(-6.0, 3.0))\n        wave = np.zeros(CLIP_SAMPLES, dtype=np.float32)\n        placed = (seg * db_to_amp(gain)).astype(np.float32)\n        peak = float(np.max(np.abs(placed))) if len(placed) else 0.0\n        if peak > 0.95:\n            placed = (placed * (0.95 / peak)).astype(np.float32)\n            print(f"speech_{i:02d}: gain {gain:.2f} dB would peak at "\n                  f"{peak:.2f}; scaled to a 0.95 peak instead")\n        s0 = int(offset * SR)\n        wave[s0:s0 + len(placed)] += placed\n        name = f"speech_{i:02d}.wav"\n        path = write_clip(name, wave)\n        rows.append({\n            "clip_name": name[:-4], "group": "speech", "file": path,\n            "freq_hz": "", "start_s": round(offset, 3),\n            "burst_s": round(len(seg) / SR, 3), "level_dbfs": "",\n            "gain_db": round(gain, 2), "prompt": PROMPTS["speech"],\n            "_wave": wave})\n    # ---- negatives (48, shared pool)\n    for i in range(12):\n        wave = np.zeros(CLIP_SAMPLES, dtype=np.float32)\n        name = f"neg_silence_{i:02d}.wav"\n        path = write_clip(name, wave)\n        rows.append({\n            "clip_name": name[:-4], "group": "negative", "file": path,\n            "freq_hz": "", "start_s": "", "burst_s": "",\n            "level_dbfs": "", "gain_db": "",\n            "prompt": PROMPTS["negative"], "_wave": wave})\n    for i in range(18):\n        level = float(rng.uniform(-30.0, -18.0))\n        burst_s = float(rng.uniform(0.5, 2.0))\n        start_s = float(rng.uniform(0.0, 4.0 - burst_s))\n        wave = np.zeros(CLIP_SAMPLES, dtype=np.float32)\n        n_burst = int(burst_s * SR)\n        noise = rng.standard_normal(n_burst).astype(np.float32)\n        cur = float(np.sqrt(np.mean(np.square(\n            noise.astype(np.float64)))))\n        noise = (noise * (db_to_amp(level) / cur)).astype(np.float32)\n        s0 = int(start_s * SR)\n        wave[s0:s0 + n_burst] += noise\n        name = f"neg_noise_{i:02d}.wav"\n        path = write_clip(name, wave)\n        rows.append({\n            "clip_name": name[:-4], "group": "negative", "file": path,\n            "freq_hz": "", "start_s": round(start_s, 3),\n            "burst_s": round(burst_s, 3), "level_dbfs": round(level, 2),\n            "gain_db": "", "prompt": PROMPTS["negative"],\n            "_wave": wave})\n    for i in range(18):\n        level = float(rng.uniform(-24.0, -16.0))\n        wave = np.zeros(CLIP_SAMPLES, dtype=np.float32)\n        t = np.arange(CLIP_SAMPLES) / SR\n        wave += (db_to_amp(level)\n                 * np.sin(2 * np.pi * 120 * t)).astype(np.float32)\n        name = f"neg_rumble_{i:02d}.wav"\n        path = write_clip(name, wave)\n        rows.append({\n            "clip_name": name[:-4], "group": "negative", "file": path,\n            "freq_hz": 120, "start_s": 0.0, "burst_s": 4.0,\n            "level_dbfs": round(level, 2), "gain_db": "",\n            "prompt": PROMPTS["negative"], "_wave": wave})\n    # ---- measure every clip before anything uses it\n    print("-" * 70)\n    print(f"generated {len(rows)} clips; measuring each one:")\n    for r in rows:\n        w = r.pop("_wave")\n        n = len(w)\n        level = rms_db_np(w)\n        print(f"  {r[\'clip_name\']:<16} group={r[\'group\']:<8} "\n              f"samples={n} duration={n / SR:.3f} s "\n              f"rms={level:.2f} dB")\n        if n != CLIP_SAMPLES:\n            print()\n            print("!" * 70)\n            print(f"CLIP MEASUREMENT FAILED: {r[\'clip_name\']} has "\n                  f"{n} samples, not {CLIP_SAMPLES}. The generated")\n            print("set is not what the plan says. Nothing was")\n            print("extracted. Show this whole output to Muse.")\n            print("!" * 70)\n            sys.exit(9)\n    def write_manifest(p):\n        with open(p, "w", newline="") as fh:\n            w = csv.DictWriter(fh, fieldnames=MANIFEST_FIELDS)\n            w.writeheader()\n            for r in rows:\n                w.writerow({k: r.get(k, "") for k in MANIFEST_FIELDS})\n    save_both("tracing_manifest.csv", write_manifest)\n    return rows\n\n# --- Extraction --------------------------------------------------------------------------\n# One frozen forward pass per clip. Per layer, the vectors at\n# the audio-token positions are averaged into one vector per\n# clip (the layer\'s sketch of the sound, in the plan\'s plain\n# words) and stored float32. The attention subset (the first\n# 6 tone and first 6 negative clips, in manifest order) gets\n# a second forward with attentions returned; for its last 4\n# layers we record the mean attention from the final sequence\n# position to the audio-token positions.\nFEATURES_NAME = "tracing_features.npz"\n\ndef find_features():\n    for p in out_paths(FEATURES_NAME):\n        if os.path.exists(p):\n            return p\n    return None\n\ndef extract(rows):\n    n_clips = len(rows)\n    feats = np.zeros((N_STATES, n_clips, HIDDEN_SIZE),\n                     dtype=np.float32)\n    audio_counts = np.zeros(n_clips, dtype=np.int64)\n    seq_lens = np.zeros(n_clips, dtype=np.int64)\n    attention_rows = []\n    subset = set()\n    tone_seen = neg_seen = 0\n    for idx, r in enumerate(rows):\n        if r["group"] == "tone" and tone_seen < 6:\n            subset.add(idx)\n            tone_seen += 1\n        elif r["group"] == "negative" and neg_seen < 6:\n            subset.add(idx)\n            neg_seen += 1\n    for idx, r in enumerate(rows):\n        expect_silence = r["clip_name"].startswith("neg_silence")\n        try:\n            inputs, audio_np = feed_inputs(\n                r["file"], r["prompt"],\n                expect_silence=expect_silence)\n        except FeedError as e:\n            print()\n            print("!" * 70)\n            print("EXTRACTION FEED HARD STOP:")\n            print(str(e))\n            print(f"Clip {idx + 1} of {n_clips} ({r[\'clip_name\']}) "\n                  f"was not extracted, and the run stops here. No")\n            print("padding was substituted. Show this whole output")\n            print("to Muse.")\n            print("!" * 70)\n            sys.exit(7)\n        row_ids = inputs.input_ids[0].tolist()\n        pos = audio_positions(row_ids)\n        if len(pos) == 0:\n            print()\n            print("!" * 70)\n            print(f"EXTRACTION HARD STOP: clip {r[\'clip_name\']} has "\n                  f"zero audio-token positions under the gate\'s")\n            print("audio token id. The gate found them on the tone")\n            print("master, so this clip\'s feed differs somehow. No")\n            print("vector was faked. Show this output to Muse.")\n            print("!" * 70)\n            sys.exit(8)\n        out = thinker_forward(inputs)\n        hs = out.hidden_states\n        pos_t = torch.tensor(pos, dtype=torch.long)\n        for li in range(N_STATES):\n            layer_vec = hs[li][0].index_select(0, pos_t.to(\n                hs[li].device)).mean(dim=0)\n            feats[li, idx] = layer_vec.float().cpu().numpy()\n        audio_counts[idx] = len(pos)\n        seq_lens[idx] = int(hs[0].shape[1])\n        print(f"extracted {idx + 1}/{n_clips}: {r[\'clip_name\']} "\n              f"(group={r[\'group\']}, audio tokens={len(pos)}, "\n              f"seq len={seq_lens[idx]})")\n        del out, inputs\n        if idx in subset:\n            inputs2, _ = feed_inputs(\n                r["file"], r["prompt"],\n                expect_silence=expect_silence)\n            out2 = thinker_forward(inputs2, want_attentions=True)\n            atts = getattr(out2, "attentions", None)\n            if atts is None or len(atts) == 0:\n                print(f"WARNING: attention subset clip "\n                      f"{r[\'clip_name\']} returned no attentions; "\n                      f"its rows will be missing from the attention")\n                print("CSV, and this warning is the record of why.")\n            else:\n                n_attn_layers = len(atts)\n                for li in range(n_attn_layers - 4, n_attn_layers):\n                    a = atts[li][0].float()  # heads, seq, seq\n                    val = float(a[:, -1, :][:, pos].mean().item())\n                    attention_rows.append({\n                        "clip": r["clip_name"], "layer": li,\n                        "attention_to_audio": round(val, 6)})\n                print(f"attention read: {r[\'clip_name\']} "\n                      f"(last 4 of {n_attn_layers} layers)")\n            del out2, inputs2\n        free_memory()\n    extracted = int(np.count_nonzero(audio_counts))\n    print(f"extraction count: {extracted} of {n_clips} clips")\n    if extracted != n_clips:\n        print()\n        print("!" * 70)\n        print(f"EXTRACTION COUNT MISMATCH: {extracted} clips have "\n              f"features, the manifest lists {n_clips}. The run")\n        print("stops; partial features are not saved as a result.")\n        print("!" * 70)\n        sys.exit(10)\n    groups = np.array([r["group"] for r in rows])\n    names = np.array([r["clip_name"] for r in rows])\n\n    def write_npz(p):\n        np.savez(p, features=feats, groups=groups, clip_names=names,\n                 audio_token_counts=audio_counts, seq_lens=seq_lens)\n    save_both(FEATURES_NAME, write_npz)\n    return feats, groups, names, attention_rows\n\n# --- Battery flow ---------------------------------------------------------------------------\nrows = generate_clips()\nattention_rows = []\nfeats = groups = names = None\nexisting = find_features()\nif existing is not None:\n    try:\n        z = np.load(existing, allow_pickle=False)\n        f0 = z["features"]\n        g0 = [str(x) for x in z["groups"]]\n        n0 = [str(x) for x in z["clip_names"]]\n        if f0.shape == (N_STATES, len(rows), HIDDEN_SIZE) \\\n                and n0 == [r["clip_name"] for r in rows] \\\n                and g0 == [r["group"] for r in rows]:\n            feats, groups, names = f0, z["groups"], z["clip_names"]\n            print(f"resume: {existing} already holds all "\n                  f"{len(rows)} clips with matching names and "\n                  f"groups; extraction skipped, going straight "\n                  f"to the probes. The attention readout is not "\n                  f"re-run on a resume; its CSV from the first "\n                  f"pass stands.")\n        else:\n            print(f"resume: {existing} exists but does not match "\n                  f"this manifest (shape {f0.shape}); extracting "\n                  f"fresh and overwriting it.")\n    except Exception as e:\n        print(f"resume: could not read {existing} ({str(e)[:120]}); "\n              f"extracting fresh.")\nif feats is None:\n    feats, groups, names, attention_rows = extract(rows)\n\nif attention_rows:\n    def write_attention(p):\n        with open(p, "w", newline="") as fh:\n            w = csv.DictWriter(fh, fieldnames=[\'clip\', \'layer\', \'attention_to_audio\'])\n            w.writeheader()\n            w.writerows(attention_rows)\n    save_both("tracing_attention.csv", write_attention)\n\n# --- The helpers (probes) ---------------------------------------------------------------------\n# One logistic regression per layer per task. The split is by\n# clip index within each class, shuffled with the run seed:\n# the first 32 of each class train, the last 16 test, the\n# same split at every layer and for both tasks, printed here\n# so it can be re-derived. The control trains on the same\n# training vectors with the labels shuffled (seed plus the\n# layer), and is tested on the true test labels: a control\n# that beats chance means the helper is reading some side\n# pattern, and the plan\'s rule voids the run.\nfrom sklearn.linear_model import LogisticRegression\n\nPROBE_FIELDS = [\'layer\', \'task\', \'n_train\', \'n_test\', \'accuracy\', \'shuffled_accuracy\']\ngroup_list = [str(g) for g in groups]\nclass_indices = {}\nfor gname in ("tone", "speech", "negative"):\n    idxs = [i for i, g in enumerate(group_list) if g == gname]\n    rng_split = np.random.default_rng(SEED)\n    order = rng_split.permutation(idxs)\n    class_indices[gname] = ([int(x) for x in order[:32]],\n                            [int(x) for x in order[32:]])\n    print(f"split for {gname}: {len(class_indices[gname][0])} "\n          f"train, {len(class_indices[gname][1])} test "\n          f"(seed {SEED})")\n\nprobe_rows = []\nfor task, pos_group in (("tone_vs_negatives", "tone"),\n                        ("speech_vs_negatives", "speech")):\n    tr_pos, te_pos = class_indices[pos_group]\n    tr_neg, te_neg = class_indices["negative"]\n    tr_idx = tr_pos + tr_neg\n    te_idx = te_pos + te_neg\n    y_train = np.array([1] * len(tr_pos) + [0] * len(tr_neg))\n    y_test = np.array([1] * len(te_pos) + [0] * len(te_neg))\n    for li in range(N_STATES):\n        X = feats[li]\n        clf = LogisticRegression(max_iter=1000, C=1.0)\n        clf.fit(X[tr_idx], y_train)\n        acc = float(clf.score(X[te_idx], y_test))\n        rng_shuf = np.random.default_rng(SEED + 1000 + li)\n        y_shuf = rng_shuf.permutation(y_train)\n        clf2 = LogisticRegression(max_iter=1000, C=1.0)\n        clf2.fit(X[tr_idx], y_shuf)\n        acc_shuf = float(clf2.score(X[te_idx], y_test))\n        probe_rows.append({\n            "layer": li, "task": task,\n            "n_train": len(tr_idx), "n_test": len(te_idx),\n            "accuracy": round(acc, 4),\n            "shuffled_accuracy": round(acc_shuf, 4)})\n        print(f"probe {task} layer {li}: accuracy {acc:.3f}, "\n              f"shuffled control {acc_shuf:.3f}")\n\ndef write_probes(p):\n    with open(p, "w", newline="") as fh:\n        w = csv.DictWriter(fh, fieldnames=PROBE_FIELDS)\n        w.writeheader()\n        w.writerows(probe_rows)\nsave_both("tracing_probe_results.csv", write_probes)\n\n# --- The plain-language tail ---------------------------------------------------------------------\nprint()\nprint("=" * 70)\nprint("READOUT (the plan\'s rules, applied mechanically)")\nfor task in ("tone_vs_negatives", "speech_vs_negatives"):\n    accs = [r["accuracy"] for r in probe_rows if r["task"] == task]\n    fade = "never"\n    for li in range(len(accs)):\n        if accs[li] < 0.70 and all(a < 0.70 for a in accs[li:]):\n            fade = str(li)\n            break\n    print(f"  {task}: first layer where the helper falls below "\n          f"0.70 and stays below: {fade}")\nshufs = [r["shuffled_accuracy"] for r in probe_rows]\nprint(f"  shuffled controls across all layers and tasks: "\n      f"min {min(shufs):.3f}, max {max(shufs):.3f} "\n      f"(chance is about 0.50; a control well above chance "\n      f"voids the run under the plan\'s rules)")\nprint("  layer 0 is the embedding output; layer k is the output")\nprint("  of thinker layer k. A fade is where this helper stops")\nprint("  reading the sound, not proof the sound is gone: the")\nprint("  surgery build is the causal check, per the plan.")\nprint("=" * 70)\nprint("battery done: clips generated and measured, features")\nprint("extracted and counted, helpers trained with controls,")\nprint("attention read on the subset. Outputs: "\n      "tracing_manifest.csv, tracing_features.npz, "\n      "tracing_probe_results.csv, tracing_attention.csv.")\n'
with open("/content/qwen_tracing_run.py", "w") as fh:
    fh.write(RUNNER_SOURCE)
print("runner written:", "/content/qwen_tracing_run.py", len(RUNNER_SOURCE), "chars")

In [ ]:
# --- Check mode: proof, candidates, sanity gate --------------------------------------
# Runs the runner under this session's own Python and streams
# its output here live. A nonzero exit code means the proof, a
# loading candidate, or the sanity gate failed, and the run
# stops in this cell. The first check run downloads the model
# files once; that takes a while and is normal.
import subprocess, sys
proc = subprocess.run([sys.executable, "/content/qwen_tracing_run.py", "check"])
print()
print("check exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE CHECK RUN FAILED. The runner's own output above says")
    print("where: a proof line, a loading candidate, or the sanity gate.")
    print("Do not run Step 4. Show this output to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: runner check failed")
print("check passed: proof lines right, model loaded, sanity gate")
print("passed with the audio tokens located and counted. Go on to Step 4.")

## Step 4 · Run the tracing battery

The next cell runs the runner in **battery mode**: proof,
candidates, and the sanity gate again, then four stages.

**Clip generation.** 144 clips are synthesized with the
fixed seed 20261103, each exactly 4.0 seconds at 16 kHz:
48 tone, 48 speech, 48 negatives (12 silence, 18 noise
bursts, 18 rumbles), with the parameters of every clip
recorded in `tracing_manifest.csv`. Every clip is measured
and printed (sample count, duration, RMS) before anything
uses it; a clip that fails its measurement stops the run.

**Extraction.** Each clip goes through the frozen model
once, sound alone, under the Series Two presence question
for its sound. Any feed that returns no audio, or decodes
to zeros on a clip that should carry sound, stops the run
on the spot (exit code 7): nothing is padded and nothing is
scored on a broken feed. Per layer, the vectors at the
audio-token positions are averaged into one vector per
clip and saved to `tracing_features.npz`. The extraction
count must equal the manifest count, 144, or the run stops
(exit code 10). If the features file already holds all 144
clips with matching names, extraction is skipped and the
run goes straight to the helpers.

**The helpers.** Per layer and per task (tone vs
negatives, speech vs negatives), a logistic regression
trains on 32 positives plus 32 negatives and is tested on
the held-out 16 plus 16, the same split at every layer.
Each helper is shadowed by a shuffled-label control on the
same split. Results land in `tracing_probe_results.csv`:
one row per layer per task, controls included.

**The attention readout.** The first 6 tone and first 6
negative clips get a second pass with attentions returned.
For the last 4 layers, the mean attention from the final
sequence position to the audio-token positions lands in
`tracing_attention.csv`.

**The tail.** The runner ends with a plain-language
readout: for each task, the first layer where the helper
falls below 0.70 and stays below (or "never"), and the
shuffled controls' range across the whole run.

In [ ]:
# --- Battery mode: the tracing round 1 battery -----------------------------------------
# Same runner, same session Python, streamed live. The runner
# saves its outputs itself, to the Drive tracing folder and
# to /content. A nonzero exit code means a gate failed, a
# feed hard-stopped the run, clip generation failed its
# measurements, or the extraction count mismatched (the
# banners above say which).
import subprocess, sys
proc = subprocess.run([sys.executable, "/content/qwen_tracing_run.py", "battery"])
print()
print("battery exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE BATTERY RUN DID NOT FINISH CLEAN. Read the runner's")
    print("output above: a gate failed, a feed proof hard-stopped the run,")
    print("clip generation failed, or the counts mismatched. Show it to")
    print("Muse.")
    print("!" * 72)
    raise SystemExit("STOP: battery run not clean")
print("battery done: features extracted and counted, helpers trained")
print("with controls, attention read, CSVs saved.")

## Closing · How to read the answers

Read each task's curve on its own first: one accuracy per
layer, from the embedding output (layer 0) to the last
thinker layer, with the shuffled control beside it.

- **The tone fades somewhere.** The fade layer is the
  neighborhood where this helper stops reading the tone.
  Read it with the attention numbers beside it: if the
  final position's attention to the tone's tokens was near
  zero all along, the answer never looked; if attention was
  healthy and the report still missed, it looked and
  dropped. Either way, the fade is a readability result,
  not yet a proof of loss. The surgery build is the causal
  check that confirms or kills the claim.
- **The speech reads at every layer.** Then nothing about
  the speech is lost inside this model, and its denial in
  the outside tests lives in how the presence answer is
  formed, past the information. That was the registered
  prediction.
- **The speech fades somewhere.** Then the prediction dies
  in public, the taxonomy changes, and the write-up says so
  plainly. A result that kills a prediction is a result.
- **A control beats chance.** The run is void until fixed,
  under the plan's rules, whatever the curves look like.
  A helper that can read shuffled labels is reading some
  side pattern in the clips, not the sound.
- **Scope.** Every claim stays capped at this model, these
  sounds, this question per class, this run. Pegasus stays
  a black box; MiniCPM and OLA tracing wait until these
  instruments have run end to end here; the picture
  condition is round 2.

When you finish, the files in the `tracing-qwen3b` Drive
folder are the record: the manifest, the features, the
probe results, and the attention readout. Send them back,
and the tracing gets written up next to the codec control.